<a href="https://colab.research.google.com/github/Abhinavthirumalaiswamy/Multiclass-fish-image-classification/blob/main/Multiclass_Fish_Image_Classification_Mini_project_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Multiclass fish image classification

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import zipfile
import os

# Paths
# Path to your zip file
zip_path = "/content/Dataset.zip"
extract_path = "/content/Dataset"


# Extract dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define base data directory (after extraction)
data_dir = os.path.join(extract_path, "images.cv_jzk6llhf18tm3k0kyttxz", "data")

# Data preprocessing & augmentation
datagen = ImageDataGenerator(rescale=1./255,rotation_range=20,zoom_range=0.2,horizontal_flip=True)

# Train / Validation / Test generators
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")
test_dir  = os.path.join(data_dir, "test")

train_gen = datagen.flow_from_directory(train_dir,target_size=(224, 224),batch_size=32,class_mode='categorical')

val_gen = datagen.flow_from_directory(val_dir,target_size=(224, 224),batch_size=32,class_mode='categorical')

test_gen = datagen.flow_from_directory(test_dir,target_size=(224, 224),batch_size=32,class_mode='categorical',shuffle=False)

# Number of classes
num_classes = train_gen.num_classes
print("Number of classes:", num_classes)

Found 6225 images belonging to 11 classes.
Found 1092 images belonging to 11 classes.
Found 3187 images belonging to 11 classes.
Number of classes: 11


In [ ]:
print(train_gen.class_indices)

# List of class names in index order
print(train_gen.classes)

{'animal fish': 0, 'animal fish bass': 1, 'fish sea_food black_sea_sprat': 2, 'fish sea_food gilt_head_bream': 3, 'fish sea_food hourse_mackerel': 4, 'fish sea_food red_mullet': 5, 'fish sea_food red_sea_bream': 6, 'fish sea_food sea_bass': 7, 'fish sea_food shrimp': 8, 'fish sea_food striped_red_mullet': 9, 'fish sea_food trout': 10}
[ 0  0  0 ... 10 10 10]


In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers

# 1. CNN from Scratch
print("Training CNN from scratch model...")
scratch_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')])

scratch_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

# Training the model
scratch_model.fit(train_gen, validation_data=val_gen, epochs=50)

Training CNN from scratch model...
Epoch 1/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 96s 481ms/step - accuracy: 0.3364 - loss: 7.8239 - val_accuracy: 0.6484 - val_loss: 1.1674
Epoch 2/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 91s 466ms/step - accuracy: 0.6867 - loss: 0.9505 - val_accuracy: 0.7381 - val_loss: 0.8264
Epoch 3/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 89s 455ms/step - accuracy: 0.7761 - loss: 0.6527 - val_accuracy: 0.8205 - val_loss: 0.5421
Epoch 4/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 89s 458ms/step - accuracy: 0.8457 - loss: 0.4626 - val_accuracy: 0.8654 - val_loss: 0.4369
Epoch 5/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 89s 457ms/step - accuracy: 0.8657 - loss: 0.3833 - val_accuracy: 0.9057 - val_loss: 0.3127
Epoch 6/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 89s 458ms/step - accuracy: 0.8912 - loss: 0.3310 - val_accuracy: 0.8919 - val_loss: 0.3724
Epoch 7/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 89s 455ms/step - accuracy: 0.8991 - loss: 0.2991 - val_accuracy: 0.9158 - val_loss: 0.2806
Epoch 8/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 90s 461m

In [ ]:
# Model Training (MobileNetV2)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers

# 1. Load Pre-trained MobileNetV2
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_mobilenet.trainable = False

# 2. Build Model Architecture
mobilenet_model = models.Sequential([
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

# 3. Compile and Train
mobilenet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
mobilenet_model.fit(train_gen, validation_data=val_gen, epochs=5)

# 4. Save the Final Model
mobilenet_model.save('best_fish_classifier_mobilenet.h5')

Epoch 1/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 126s 594ms/step - accuracy: 0.7268 - loss: 0.8473 - val_accuracy: 0.9441 - val_loss: 0.1590
Epoch 2/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 92s 468ms/step - accuracy: 0.9550 - loss: 0.1374 - val_accuracy: 0.9835 - val_loss: 0.0623
Epoch 3/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 90s 461ms/step - accuracy: 0.9669 - loss: 0.0979 - val_accuracy: 0.9799 - val_loss: 0.0645
Epoch 4/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 91s 465ms/step - accuracy: 0.9799 - loss: 0.0639 - val_accuracy: 0.9872 - val_loss: 0.0393
Epoch 5/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 90s 461ms/step - accuracy: 0.9810 - loss: 0.0564 - val_accuracy: 0.9918 - val_loss: 0.0347


In [ ]:
# Model Training (EfficientNetB0)

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import models, layers

# 1. Load Pre-trained EfficientNetB0
base_efficient = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_efficient.trainable = False

# 2. Build Model Architecture
efficient_model = models.Sequential([
    base_efficient,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation='softmax')])

# 3. Compile and Train
efficient_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
efficient_model.fit(train_gen, validation_data=val_gen, epochs=5)

# 4. Save the Final Model
efficient_model.save('best_fish_classifier_efficientnet.h5')

Epoch 1/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 641s 3s/step - accuracy: 0.1409 - loss: 2.3530 - val_accuracy: 0.1712 - val_loss: 2.3144
Epoch 2/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 621s 3s/step - accuracy: 0.1672 - loss: 2.3109 - val_accuracy: 0.1712 - val_loss: 2.3129
Epoch 3/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 621s 3s/step - accuracy: 0.1794 - loss: 2.3026 - val_accuracy: 0.1712 - val_loss: 2.3134
Epoch 4/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 622s 3s/step - accuracy: 0.1762 - loss: 2.3020 - val_accuracy: 0.1712 - val_loss: 2.3109
Epoch 5/5
195/195 ━━━━━━━━━━━━━━━━━━━━ 621s 3s/step - accuracy: 0.1799 - loss: 2.3030 - val_accuracy: 0.1712 - val_loss: 2.3116


In [ ]:
# Model Evaluation

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Plot Training History
def plot_history(history, model_name):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy Plot
    ax[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
    ax[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
    ax[0].set_title(f'{model_name} - Model Accuracy')
    ax[0].set_ylabel('Accuracy')
    ax[0].set_xlabel('Epoch')
    ax[0].legend()

    # Loss Plot
    ax[1].plot(history.history['loss'], label='Train Loss', color='blue')
    ax[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
    ax[1].set_title(f'{model_name} - Model Loss')
    ax[1].set_ylabel('Loss')
    ax[1].set_xlabel('Epoch')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

# 2. Evaluate Performance & Confusion Matrix
def evaluate_performance(model, generator):
    generator.reset()
    labels = list(generator.class_indices.keys())

    # Generate Predictions
    preds = model.predict(generator, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = generator.classes

    # Print Metrics Report
    print(f"\n--- Classification Report: {model.name} ---")
    print(classification_report(y_true, y_pred, target_names=labels))

    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix - {model.name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()


Execution

Test

In [ ]:
# Multiclass fish image classification

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import zipfile
import os

# Paths
# Path to your zip file
zip_path = "/content/Dataset.zip"
extract_path = "/content/Dataset"


# Extract dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define base data directory (after extraction)
data_dir = os.path.join(extract_path, "images.cv_jzk6llhf18tm3k0kyttxz", "data")

# Data preprocessing & augmentation
datagen = ImageDataGenerator(rescale=1./255,rotation_range=20,zoom_range=0.2,horizontal_flip=True)

# Train / Validation / Test generators
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")
test_dir  = os.path.join(data_dir, "test")

train_gen = datagen.flow_from_directory(train_dir,target_size=(224, 224),batch_size=32,class_mode='categorical')

val_gen = datagen.flow_from_directory(val_dir,target_size=(224, 224),batch_size=32,class_mode='categorical')

test_gen = datagen.flow_from_directory(test_dir,target_size=(224, 224),batch_size=32,class_mode='categorical',shuffle=False)

# Number of classes
num_classes = train_gen.num_classes
print("Number of classes:", num_classes)

Found 6225 images belonging to 11 classes.
Found 1092 images belonging to 11 classes.
Found 3187 images belonging to 11 classes.
Number of classes: 11


In [ ]:
num_classes=train_gen.num_classes
print(num_classes)

11


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import layers, models

# 1. Setup Model
num_classes =num_classes
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

model = models.Sequential([base,layers.GlobalAveragePooling2D(),layers.Dense(128, activation='relu'),layers.Dropout(0.3),
layers.Dense(num_classes, activation='softmax')])

# 2. Load Weights
model.load_weights(r"/content/best_fish_classifier_mobilenet.h5")

# 3. Process Image
img = tf.keras.utils.load_img(r"/content/0FKL19F3O23W.jpg", target_size=(224, 224))
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# 4. Predict
pred = model.predict(img_array)
pred_class = np.argmax(pred, axis=1)[0]

print(f"Predicted Class Index: {pred_class}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted Class Index: 4
